In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

##Source path

In [0]:
SOURCE_PATH = "/Volumes/workspace/bronze/banking_source/transactions"

BRONZE_TABLE = "workspace.bronze.transactions"

print("Source:", SOURCE_PATH)
print("Target:", BRONZE_TABLE)

Source: /Volumes/workspace/bronze/banking_source/transactions
Target: workspace.bronze.transactions


##Read the CSV files

In [0]:
raw_transactions = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(f"{SOURCE_PATH}/*.csv")
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file")
    )
)

print(
    "Source records:",
    f"{raw_transactions.count():,}"
)

Source records: 5,010,000


##Inspect schema

In [0]:
raw_transactions.printSchema()

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- txn_time: timestamp (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _source_file: string (nullable = false)



##Bronze audit metadata

In [0]:
bronze_transactions = (
    raw_transactions
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_system", F.lit("BANKING_TRANSACTION_SOURCE"))
)

##Write Bronze Delta table

In [0]:
(
    bronze_transactions
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print("Bronze transaction table created")

Bronze transaction table created


##Verify Bronze

In [0]:
bronze_count = spark.table(BRONZE_TABLE).count()

print(
    "Bronze transaction records:",
    f"{bronze_count:,}"
)

Bronze transaction records: 5,010,000


In [0]:
display(
    spark.table(BRONZE_TABLE)
    .limit(20)
)

txn_id,account_id,customer_id,merchant_id,device_id,txn_time,txn_type,amount,currency,channel,status,_source_file,_ingest_ts,_source_system
TXN0000001172,ACC001280,CUST085858,MER08562,DEV016403,2025-12-19T11:17:02.000Z,DEBIT,46019.53,INR,CARD,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE
TXN0000003722,ACC004108,CUST084372,MER02147,DEV052005,2025-12-22T12:08:58.000Z,CREDIT,378420.73,INR,NETBANKING,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE
TXN0000006272,ACC006937,CUST015690,MER20758,DEV012714,2025-05-29T00:16:00.000Z,DEBIT,320072.97,INR,NETBANKING,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE
TXN0000008822,ACC009779,CUST002544,MER14382,DEV048339,2025-09-07T07:34:37.000Z,DEBIT,28044.27,INR,UPI,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE
TXN0000011372,ACC012610,CUST068717,MER07975,DEV009032,2025-04-28T19:23:45.000Z,CREDIT,12610.93,INR,CARD,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE
TXN0000013922,ACC015437,CUST093975,MER01569,DEV044662,2025-11-07T02:57:23.000Z,CREDIT,126591.15,INR,CARD,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE
TXN0000016472,ACC018255,CUST054573,MER20180,DEV005368,2025-09-23T22:17:44.000Z,DEBIT,9313.15,INR,UPI,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE
TXN0000019022,ACC021075,CUST012747,MER13805,DEV041009,2025-12-27T04:42:00.000Z,DEBIT,96361.89,INR,MOBILE,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE
TXN0000021572,ACC023883,CUST046608,MER07391,DEV001684,2025-12-15T13:30:09.000Z,CREDIT,16418.03,INR,NETBANKING,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE
TXN0000024122,ACC026748,CUST061844,MER00995,DEV037342,2025-02-25T23:49:15.000Z,DEBIT,110300.65,INR,CARD,SUCCESS,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,2026-08-22T06:50:53.670Z,BANKING_TRANSACTION_SOURCE


##Verify duplicate transactions

In [0]:
duplicate_count = (
    spark.table(BRONZE_TABLE)
    .groupBy("txn_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Transaction IDs having duplicates:",
    f"{duplicate_count:,}"
)

Transaction IDs having duplicates: 10,000


In [0]:
display(
    spark.table(BRONZE_TABLE)
    .select(
        "txn_id",
        "_ingest_ts",
        "_source_file",
        "_source_system"
    )
    .limit(20)
)

txn_id,_ingest_ts,_source_file,_source_system
TXN0000001172,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
TXN0000003722,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
TXN0000006272,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
TXN0000008822,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
TXN0000011372,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
TXN0000013922,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
TXN0000016472,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
TXN0000019022,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
TXN0000021572,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
TXN0000024122,2026-08-22T06:50:53.670Z,dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00026-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2910-1-c000.csv,BANKING_TRANSACTION_SOURCE
